In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import metrics
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import sklearn
import matplotlib.pyplot as plt

In [2]:
feature_names = ['Age', 'Gender', 'Air Pollution', 'Alcohol Usage', 'Genetic Risk', 'Lung Disease', 'Obesity', 'Smoking', 'Passive Smoker', 'Chest Pain', 'Coughing of Blood', 'Severity']
df = pd.read_excel('..\\..\\Datasets\\NewDataset\\BlackLineHospitalLungCancerDataset.xlsx', header=0) 

X = df.drop(['Severity'], axis=1)
y = df['Severity']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)

npX_train = X_train.values
npX_test = X_test.values
npy_train = y_train.values
npy_test = y_test.values

#These ranges were manually shortened over time to save time and find the optimal combination
param_grid = {
    'n_d': list(range(42, 43, 1)),
    'n_a': list(range(58, 64, 1)),
    'n_steps': list(range(1, 2, 1)),
    'optimizer_fn': [torch.optim.NAdam, ],
    'optimizer_params': [dict(lr=0.025), dict(lr=0.0325), dict(lr=0.030), dict(lr=0.0275),],
}



grid = sklearn.model_selection.GridSearchCV(estimator=TabNetClassifier(n_d=8, n_a=5, n_steps=4, verbose=0), param_grid=param_grid, scoring='accuracy', n_jobs=-1, cv=3)
grid_result = grid.fit(npX_train, npy_train, batch_size=220, virtual_batch_size=20,max_epochs=220)

# Printed out tuning results
#print("Best score: ", grid_result.best_score_)
#print("Best params: ", grid_result.best_params_)

optimalN_a = grid_result.best_params_['n_a']
optimalN_d = grid_result.best_params_['n_d']
optimalN_steps = grid_result.best_params_['n_steps']
optimalOptimzer_fn = grid_result.best_params_['optimizer_fn']
optimalOptimzer_params = dict(lr=grid_result.best_params_['optimizer_params']['lr'])


In [3]:
sum = 0
Trials = 10
for i in range(Trials):
    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size = .2, shuffle=True)
    npX_train = X_train.values
    npX_test = X_test.values
    npy_train = y_train.values
    npy_test = y_test.values

    clf = TabNetClassifier(n_a=optimalN_a, n_d=optimalN_d, n_steps=optimalN_steps, optimizer_fn=optimalOptimzer_fn, optimizer_params=optimalOptimzer_params, verbose=0)
    clf.fit(npX_train, npy_train, batch_size=100, virtual_batch_size=20, max_epochs=220, eval_metric = ['accuracy'], compute_importance=True) 
    featureimportances = clf.feature_importances_
    
    preds = clf.predict(npX_test)

    accuracy = metrics.accuracy_score(y_true=y_test, y_pred=preds)
    print(f"Trial {i} accuracy: {accuracy}")
    sum += accuracy

average = sum/Trials
print(f"\nAverage accuracy: {average}\n")

# This displays feature importance for TabNet

# indices = np.argsort(featureimportances) [::-1]
# names = [feature_names[i] for i in indices]
# plt.figure(figsize=(10,6))
# plt.title("Feature Importances of Last Trial")
# plt.bar(range(X.shape[1]), featureimportances[indices])
# plt.xticks(range(X.shape[1]), names, rotation=90)
# plt.xlabel("Features")
# plt.ylabel("Importance")
# plt.show()


Trial 0 accuracy: 0.9897610921501706
Trial 1 accuracy: 0.9829351535836177
Trial 2 accuracy: 0.9931740614334471
Trial 3 accuracy: 0.9829351535836177
Trial 4 accuracy: 0.9863481228668942
Trial 5 accuracy: 0.9863481228668942
Trial 6 accuracy: 0.9863481228668942
Trial 7 accuracy: 0.9863481228668942
Trial 8 accuracy: 0.9897610921501706
Trial 9 accuracy: 0.9863481228668942

Average accuracy: 0.9870307167235495

